In [1]:
import tensorflow as tf
import os
import numpy as np
from tensorflow.keras import models,layers
from tensorflow.keras.callbacks import EarlyStopping
import json

In [2]:
def model_make(path):
    batch=32
    image_size=256
    channel=3
    dataset_name = os.path.basename(path)
    print("-"*50,dataset_name)
    # train,test,val split 
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=(256,256),
        batch_size=batch
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=(256,256),
        batch_size=batch
    )
    # checking class names 
    class_name = train_ds.class_names
    print(class_name)
    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
    # ---------------resize and rescale--------------
    image_size=224
    resize_and_rescale =tf.keras.Sequential(
        [layers.Resizing(image_size,image_size),
        layers.Rescaling(1./255)]
    )
    # --------------data argumentation----------------
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
    ])
    # ---------------------model--------------------
    model = tf.keras.Sequential([
        resize_and_rescale,
        data_augmentation,
        layers.Conv2D(32,kernel_size=(3,3),input_shape=(224,224,channel),activation='relu'),
        layers.MaxPool2D((2,2)),
        layers.Conv2D(16,kernel_size=(3,3),activation='relu'),
        layers.MaxPool2D((2,2)),
        layers.Conv2D(8,kernel_size=(3,3),activation='relu'),
        layers.MaxPool2D((2,2)),
        layers.Flatten(),
        layers.Dense(64,activation='relu'),
        layers.Dense(len(class_name), activation='softmax')
    ])
    model.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
    )
    # early stopping and modelcheckpoint implementation
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,  
        restore_best_weights=True
    )
    # model training
    history = model.fit(
        train_ds,
        epochs=500, 
        validation_data=val_ds,
        callbacks = [early_stop]
    )
    # model evaluation
    print("-"*50)
    print(history.history['accuracy'])
    print(history.history['val_accuracy'])
    print("-"*50)
    model.evaluate(val_ds)  
    model.save(f"{dataset_name}.keras")  
    with open(f"{dataset_name}_classes.json", "w") as f:
        json.dump(class_name, f)    
    return model

        

In [3]:
crop_classification='Dataset_path'
model_make(crop_classification)

-------------------------------------------------- crop_classification
Found 3115 files belonging to 6 classes.
Using 2492 files for training.
Found 3115 files belonging to 6 classes.
Using 623 files for validation.
['banana', 'grapes', 'mango', 'potato', 'strawberry', 'tomato']
Epoch 1/500


c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


78/78 ━━━━━━━━━━━━━━━━━━━━ 30s 311ms/step - accuracy: 0.4510 - loss: 1.2905 - val_accuracy: 0.6404 - val_loss: 0.9668
Epoch 2/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 25s 317ms/step - accuracy: 0.6762 - loss: 0.8066 - val_accuracy: 0.7592 - val_loss: 0.6675
Epoch 3/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 24s 304ms/step - accuracy: 0.7488 - loss: 0.6507 - val_accuracy: 0.7352 - val_loss: 0.6406
Epoch 4/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 74s 956ms/step - accuracy: 0.7825 - loss: 0.5599 - val_accuracy: 0.7560 - val_loss: 0.6118
Epoch 5/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 73s 929ms/step - accuracy: 0.7889 - loss: 0.5439 - val_accuracy: 0.8186 - val_loss: 0.4899
Epoch 6/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 25s 325ms/step - accuracy: 0.8407 - loss: 0.4570 - val_accuracy: 0.8074 - val_loss: 0.4688
Epoch 7/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 25s 315ms/step - accuracy: 0.8543 - loss: 0.3981 - val_accuracy: 0.8700 - val_loss: 0.3795
Epoch 8/500
78/78 ━━━━━━━━━━━━━━━━━━━━ 25s 322ms/step - accuracy: 0.8543 - loss: 0.3898 - val_accuracy

<Sequential name=sequential_2, built=True>